In [ ]:
import re
import subprocess
from pathlib import Path

CODE_REVISION = "ВСТАВ_SHA_НОВОГО_КОМІТУ"
PROJECT_DIR = Path("/kaggle/working/drone_detection")

if not re.fullmatch(r"[0-9a-f]{40}", CODE_REVISION):
    raise ValueError("Set CODE_REVISION to the full project commit SHA")

if not PROJECT_DIR.exists():
    subprocess.run([
        "git", "clone",
        "https://github.com/Hannysia/drone_detection.git",
        str(PROJECT_DIR),
    ], check=True)

subprocess.run([
    "git", "-C", str(PROJECT_DIR),
    "fetch", "origin", CODE_REVISION,
], check=True)

subprocess.run([
    "git", "-C", str(PROJECT_DIR),
    "checkout", "--detach", CODE_REVISION,
], check=True)

%pip install -q -e /kaggle/working/drone_detection timm huggingface_hub
%cd /kaggle/working/drone_detection

In [ ]:
import gc
import hashlib
import json
from collections import Counter

import torch
import torchvision

from src.config import MANIFESTS_DIR
from src.processing.embeddings import (
    DroneEmbeddingDataset,
    generate_and_save_embeddings,
)
from src.processing.manifests import read_manifest
from src.utils.io import check_extraction_source

DATASET_ROOT = Path("ВСТАВ_ШЛЯХ_ДО_НАБОРУ_З_INPUT")
MANIFEST = DATASET_ROOT / "manifest.csv"
BATCH_SIZE = 64
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DINOV2_REVISION = "7764ea0f912e53c92e82eb78a2a1631e92725fc8"
MODEL_NAME = "dinov2_vitb14"

source = json.loads(
    (MANIFESTS_DIR / "sources" / "hf.json").read_text()
)
check_extraction_source(DATASET_ROOT, source)

version = source["annotation_version"]
rows = read_manifest(MANIFEST)
expected = read_manifest(
    MANIFESTS_DIR / "baseline" / version / "main.csv"
)

if rows != expected:
    raise ValueError("Uploaded manifest differs from the project baseline")

OUTPUT_DIR = Path("/kaggle/working/embeddings") / version

for category in ("positive", "negative"):
    dataset = DroneEmbeddingDataset(DATASET_ROOT, MANIFEST, category)
    print(f"train/{category}: {len(dataset)}")

print("Device:", DEVICE)
print("Source revision:", source["revision"])
print(
    "Manifest SHA-256:",
    hashlib.sha256(MANIFEST.read_bytes()).hexdigest(),
)

In [ ]:
if DEVICE.type != "cuda":
    raise RuntimeError(
        "Enable a GPU accelerator before generating embeddings"
    )

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

metadata = {
    "source": source,
    "manifest_sha256": hashlib.sha256(
        MANIFEST.read_bytes()
    ).hexdigest(),
    "code_revision": CODE_REVISION,
    "model": MODEL_NAME,
    "model_code_revision": DINOV2_REVISION,
    "torch": torch.__version__,
    "torchvision": torchvision.__version__,
    "batch_size": BATCH_SIZE,
    "counts": dict(Counter(
        row["category"]
        for row in rows
        if row["split"] == "train" and not row["excluded_at"]
    )),
}

metadata_path = OUTPUT_DIR / "run.json"

if metadata_path.exists():
    if json.loads(metadata_path.read_text()) != metadata:
        raise ValueError(
            "Existing embeddings belong to a different run"
        )
else:
    if any(OUTPUT_DIR.iterdir()):
        raise ValueError(
            "Output directory has files without run.json"
        )

    metadata_path.write_text(
        json.dumps(metadata, indent=2) + "\n",
        encoding="utf-8",
    )

model = torch.hub.load(
    f"facebookresearch/dinov2:{DINOV2_REVISION}",
    MODEL_NAME,
    trust_repo=True,
).to(DEVICE)

try:
    for category, suffix in (
        ("positive", "positives"),
        ("negative", "negatives"),
    ):
        output_path = OUTPUT_DIR / f"train_{suffix}_embeddings.pt"

        if output_path.exists():
            existing = torch.load(
                output_path,
                map_location="cpu",
                weights_only=False,
            )
            dataset = DroneEmbeddingDataset(
                DATASET_ROOT, MANIFEST, category,
            )

            if set(existing) != {
                path.name for path in dataset.image_paths
            }:
                raise ValueError(
                    f"Embedding filenames mismatch: {output_path}"
                )

            del existing
            print("Already completed:", output_path)
            continue

        output = generate_and_save_embeddings(
            dataset_root=DATASET_ROOT,
            manifest_path=MANIFEST,
            model=model,
            category=category,
            output_path=output_path,
            batch_size=BATCH_SIZE,
            device=DEVICE,
        )
        print("Saved:", output)
finally:
    del model
    gc.collect()
    torch.cuda.empty_cache()

Loading DINOv2 ViT-B/14 model on GPU...


Using cache found in /root/.cache/torch/hub/facebookresearch_dinov2_main
Exception in thread QueueFeederThread:
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/queues.py", line 259, in _feed
Exception ignored in: <function _ConnectionBase.__del__ at 0x7f13f3e0f560>
Traceback (most recent call last):
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 133, in __del__
    reader_close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 178, in close
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    self._close()
  File "/usr/lib/python3.12/multiprocessing/connection.py", line 377, in _close
    _close(self._handle)
OSError: [Errno 9] Bad file descriptor

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/lib/python3.12/threading.py", line 1075, in _bootstrap_inner
    _close(self._handle)
OSError: [Errno 9] Bad fi


GENERATING EMBEDDINGS FOR: TRAIN (RAW_NEGATIVES)
Reading directly from: /kaggle/input/datasets/hannusiaknysh/train-drone-detection/extracted_dataset/train

Extracting embeddings via DINOv2 model...


DINOv2 Inference:   0%|          | 0/93 [00:00<?, ?it/s]


Saving embeddings dictionary to /kaggle/working/embeddings/train_negatives_embeddings.pt...

Uploading resulting embeddings file 'train_negatives_embeddings.pt' to Hugging Face...


Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            


SUCCESS! EMBEDDINGS EXTRACTED AND UPLOADED TO HUGGING FACE!
